# Controlled improvements v009

Conditional follow-up under standing user approval: shared breath offset/centered local correction and full-context residual TCN. Start only if v008 selects no candidate; otherwise confirm the v008 winner first. Original grouped A split, combined47, seed20261102, batch256 and full200 replay rates remain fixed. Default is read-only. Main metric stays native inspiratory MAE; accepted .18550 CSV is preserved.

Actual-course CUDA determinism/resume/profile proofs precede fitting. Compare fixed equal pairs with the unchanged deep A model, retain the .003 gain plus negative bootstrap gate, and require separate fresh grouped confirmation before promotion. Modeling code is entirely in notebooks.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import copy,gc,json,hashlib,time,fcntl,uuid,platform
from pathlib import Path
from datetime import datetime,timezone
import numpy as np,pandas as pd
RUN=os.environ.get('PA3_RUN_V009','0')=='1';RESUME=os.environ.get('PA3_RESUME_V009','0')=='1'
PROJECT=Path.cwd().resolve();BASE=PROJECT/'outputs/improvements_v005/sequence';PRIOR=PROJECT/'outputs/improvements_v005/confirmation_round1';OUT=PROJECT/'outputs/controlled_experiments_v009';SEED=20261102
DTYPES=dict(id='int64',breath_id='int32',R='int8',C='int8',time_step='float32',u_in='float32',u_out='int8',pressure='float32')
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def sig(value):return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()
def ah(value):return hashlib.sha256(np.ascontiguousarray(value).tobytes()).hexdigest()
def codehash(path):
 n=json.loads(Path(path).read_text());return hashlib.sha256('\n'.join(''.join(c['source']) for c in n['cells'] if c['cell_type']=='code').encode()).hexdigest()
def local(value):
 s=str(value);marker='/ventilator-pressure-prediction/';p=PROJECT/s.split(marker,1)[1] if marker in s else PROJECT/s
 assert p.resolve().is_relative_to(PROJECT);return p

def write_json(path,value):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp-'+uuid.uuid4().hex);tmp.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n');os.replace(tmp,path)
def save_array(path,a):
 path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
 with tmp.open('wb') as f:np.save(f,a,allow_pickle=False)
 os.replace(tmp,path)
def event(event_name,**kw):
 with (OUT/'events.jsonl').open('a') as f:f.write(json.dumps(dict(time_utc=datetime.now(timezone.utc).isoformat(),event=event_name,**kw))+'\n')
COMPONENTS={'feature_components_v005.ipynb': 'eb19eed69eb54cd5d087a9a678cf98701fbe9bc1a3cf85016270754c014593a6', 'sequence_components_v005.ipynb': 'e6698a7f6e931af8a066c227058203b7e29ada68624833b700293532a30bf1b2', 'training_components_v008.ipynb': 'a65618d1886a7ad3d1753f1f752b449114a57dcb75084f4f6cd11fc084228bda', 'offset_components_v009.ipynb': '1000c5311fcf4cfc348fd5b99975fcd5816650d1616ace31eace5407ccd2a1b3', 'tcn_components_v009.ipynb': 'd4e08e56a78a8d1d8e45ee0843302d7d2a1c9b35ccc1ed8628ae4f1239da486e', 'tcn_fast_components_v009.ipynb': '51d7438d2bd719e5609ed8530a4ad9cf9009943498641bf9488c9e81b1a5024d'}

ARMS=['shared_breath_offset','full_context_tcn']
LOSS_KINDS={arm:'l1' for arm in ARMS}
if RUN:
 for filename,expected in COMPONENTS.items():
  path=PROJECT/'research'/filename;assert codehash(path)==expected,'Component source changed: '+filename
  nb=json.loads(path.read_text())
  for c in nb['cells']:
   if c['cell_type']=='code':exec(compile(''.join(c['source']),filename,'exec'),globals())
 torch.set_num_threads(4);assert torch.cuda.is_available();OUT.mkdir(parents=True,exist_ok=True)
 LOCK=(OUT/'.run.lock').open('a+');fcntl.flock(LOCK,fcntl.LOCK_EX|fcntl.LOCK_NB)
 previous=json.loads((PROJECT/'outputs/controlled_experiments_v008/screen_summary.json').read_text());assert previous['status']=='complete' and previous['selected_for_confirmation'] is None,'V008 winner must be confirmed first'
 COMPACT=json.loads((BASE/'record_A_capacity_bilstm128.json').read_text());DEEP=json.loads((BASE/'record_A_capacity_deep_bilstm.json').read_text())
 SPEC=COMPACT['recipe']['model_spec'];CFG=COMPACT['recipe']['train_cfg'];assert CFG['max_epochs']==200 and CFG['batch_size']==256 and len(CFG['frozen_lr_schedule'])==200

def factory_for(arm):
 if arm=='baseline_control':return lambda spec,n_features,prep:V5PressureRNN(spec,n_features)
 if arm=='shared_breath_offset':return lambda spec,n_features,prep:v9_offset_model(spec,n_features,prep)
 if arm=='full_context_tcn':return lambda spec,n_features,prep:v9_fast_tcn_model(spec,n_features,prep)
 raise ValueError('Unknown arm: '+arm)


## Synthetic proofs and profiling

Common initialization applies only to the offset branch. TCN is a different encoder/head package with its own initialization and parameter count. Both use original batch256 and inference512, strict deterministic execution and exact interrupted-resume proofs. No target labels enter either prediction branch.


In [ ]:
def profile_v009(factory,spec,device='cuda',batch_size=256,loss_kind='l1'):
 seed_sequence(SEED)
 names=feature_names_v005('combined');rng=np.random.default_rng(8008);raw=rng.normal(size=(batch_size,80,47)).astype(np.float32)
 for k,levels in [('R',[5,20,50]),('C',[10,20,50])]:raw[:,:,names.index(k)]=rng.choice(levels,size=(batch_size,1))
 raw[:,:,names.index('u_in')]=rng.uniform(0,100,(batch_size,80));raw[:,:,names.index('dt')]=.03;raw[:,0,names.index('dt')]=0
 y=rng.normal(size=(batch_size,80)).astype(np.float32);out=np.zeros_like(y,dtype=np.int8);out[:,30:]=1
 raw[:,:,names.index('u_out')]=out;raw[:,:,names.index('time_step')]=np.cumsum(raw[:,:,names.index('dt')],axis=1);prep=fit_preprocessor(raw,y,out,spec,names);z=torch.from_numpy(transform_inputs(raw,prep)).to(device);target=torch.from_numpy(_scaled_targets(raw,y,prep)).to(device);mask=torch.from_numpy(out==0).to(device)
 model=factory(spec,47,prep).to(device);opt=torch.optim.Adam(model.parameters(),lr=.0015)
 def step():
  model.train();opt.zero_grad(set_to_none=True);p=model(z);loss=(p-target).abs().masked_select(mask).mean()
  if loss_kind=='curve_l1':loss=loss+.10*((p[:,1:]-p[:,:-1])-(target[:,1:]-target[:,:-1])).abs().masked_select(mask[:,1:] & mask[:,:-1]).mean()
  loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
 for _ in range(4):step()
 torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats();t=time.monotonic()
 for _ in range(12):step()
 torch.cuda.synchronize();seconds=(time.monotonic()-t)/12;nparams=int(sum(p.numel() for p in model.parameters()));inference_z=torch.cat([z,z],dim=0);model.eval();t=time.monotonic()
 with torch.no_grad():p=model(inference_z)
 torch.cuda.synchronize();inference_seconds=time.monotonic()-t;assert p.shape==(2*batch_size,80) and bool(torch.isfinite(p).all())
 peak=int(torch.cuda.max_memory_allocated());reserved=int(torch.cuda.max_memory_reserved());total=int(torch.cuda.get_device_properties(0).total_memory)
 result=dict(batch_size=batch_size,loss_kind=loss_kind,steps=12,step_seconds=seconds,inference_batch_size=2*batch_size,inference_seconds=inference_seconds,peak_allocated_bytes=peak,peak_reserved_bytes=reserved,total_device_bytes=total,parameter_count=nparams,strict_determinism=torch.are_deterministic_algorithms_enabled())
 del model,opt,z,target,mask,inference_z,p;gc.collect();torch.cuda.empty_cache();return result

if RUN:
 proofs={}
 proofs['offset']=v9_offset_selfcheck(device='cuda')
 proofs['tcn']=v9_fast_tcn_selfcheck(device='cuda')
 for arm in ARMS:
  proofs[arm+'_resume']=v8_training_selfcheck(factory_for(arm),arm=arm,loss_kind=LOSS_KINDS[arm],source_fingerprints=COMPONENTS,device='cuda',feature_names=feature_names_v005('combined'),n_features=47)
 write_json(OUT/'course_proofs.json',proofs)
 baseline=profile_v009(factory_for('baseline_control'),SPEC);profiles={'baseline':baseline}
 for arm in ARMS:
  try:result=profile_v009(factory_for(arm),SPEC,loss_kind=LOSS_KINDS[arm])
  except Exception as exc:
   profiles[arm]=dict(passed=False,error_type=type(exc).__name__,message=str(exc));write_json(OUT/'course_capacity_profiles.json',profiles);raise
  result['relative_step_seconds']=result['step_seconds']/baseline['step_seconds'];result['passed']=bool(result['relative_step_seconds']<=2. and result['peak_allocated_bytes']<9*1024**3);profiles[arm]=result
 write_json(OUT/'course_capacity_profiles.json',profiles)
 assert all(profiles[arm]['passed'] for arm in ARMS),'Capacity gate failed; review the saved profile before any approved-data fitting'
 event('course_synthetic_proofs_passed',profiles=profiles)


## Immutable data, split and controls

Approved Gym train file is hash checked. Complete80-point breaths remain grouped. Original A validation never enters feature fitting, target scaling or gradient updates. Repeated use of A for research limits its independence; promising choices require fresh grouped resampling confirmation.


In [ ]:
if RUN:
 manifest=json.loads((PRIOR/'final_all/final_manifest.json').read_text());assert sha(PROJECT/'data/train.csv')==manifest['train_sha256']
 base_design=json.loads((BASE/'design.json').read_text());assert codehash(PROJECT/'iterative_improvements_v005.ipynb')==base_design['source_code_sha256']
 raw=pd.read_csv(PROJECT/'data/train.csv',dtype=DTYPES);BREATH_IDS=_validate_feature_frame_v005(raw);assert len(BREATH_IDS)==67905
 Y=raw.pressure.to_numpy(copy=True).reshape(-1,80);UOUT=raw.u_out.to_numpy(copy=True).reshape(-1,80);IDS=raw.id.to_numpy(copy=True).reshape(-1,80);RC=raw[['R','C']].to_numpy(copy=True)[::80]
 frame=raw[['id','breath_id','R','C','time_step','u_in','u_out']].copy();del raw;gc.collect()
 with np.load(BASE/'split_breath_ids.npz') as z:GROUPS={k:np.sort(pd.Index(BREATH_IDS).get_indexer(z[k])) for k in z.files}
 for k,idx in GROUPS.items():assert (idx>=0).all() and len(np.unique(idx))==len(idx) and ah(BREATH_IDS[idx])==base_design['group_hashes'][k]
 fit=GROUPS['A_fit'];val=GROUPS['A_validation'];assert len(fit)==61114 and len(val)==6791 and not np.intersect1d(fit,val).size
 MASK=UOUT[val]==0;TRUTH=Y[val][MASK].astype(np.float64);VAL_IDS=IDS[val][MASK];assert np.array_equal(VAL_IDS,np.load(BASE/'A_validation_row_ids.npy'))
 X,NAMES,xids=make_sequence_features(frame,'combined');assert np.array_equal(xids,BREATH_IDS) and X.shape==(67905,80,47)
 for r in [COMPACT,DEEP]:assert sha(local(r['checkpoint_path']))==r['checkpoint_sha256'] and ah(np.load(local(r['prediction_file'])))==r['prediction_sha256']
 P_COMPACT=np.load(local(COMPACT['prediction_file'])).astype(np.float64);P_DEEP=np.load(local(DEEP['prediction_file'])).astype(np.float64);P_REFERENCE=.5*(P_COMPACT+P_DEEP)
 assert abs(float(np.abs(P_REFERENCE-TRUTH).mean())-.19656260982217663)<1e-12
 approval=json.loads((PROJECT/'research/authorization_v009.json').read_text());assert approval['standing_approval_for_controlled_iterations'] and approval['only_assignment_approved_data']
 plan=dict(version='v009-r1',arms=ARMS,loss_kinds=LOSS_KINDS,offset_hidden=16,tcn_channels=96,tcn_dilations=[1,2,4,8,16,32],tcn_backend='shifted_linear',tcn_blocks=6,tcn_kernel=3,tcn_convolutions_per_block=2,tcn_receptive_field=253,seed=SEED,group_hashes=base_design['group_hashes'],A_fit_breaths=61114,A_val_breaths=6791,features='combined47',epochs=200,full_original_schedule=True,pair_weights=[.5,.5],partner='unchanged_original_deep',minimum_pair_gain=.003,bootstrap_seed=20261201,bootstrap_draws=1000,confirmation_seed=20261202,final_fits_in_this_notebook=False,automatic_submission=False)
 design=dict(plan=plan,authorization_sha256=sha(PROJECT/'research/authorization_v009.json'),source_sha256=codehash(PROJECT/'controlled_experiments_v009.ipynb'),component_hashes=COMPONENTS,train_sha256=manifest['train_sha256'],baseline_records={k:sha(BASE/('record_A_'+k+'.json')) for k in ['capacity_bilstm128','capacity_deep_bilstm']})
 if (OUT/'design.json').exists():assert RESUME and json.loads((OUT/'design.json').read_text())==design
 else:write_json(OUT/'design.json',design)
 save_array(OUT/'A_row_ids.npy',VAL_IDS);RECORDS={}
 event('environment',pod=os.environ.get('HOSTNAME'),gpu=torch.cuda.get_device_name(0),torch=torch.__version__,cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),numpy=np.__version__,pandas=pd.__version__,design_sha256=sig(design))


## Separate fixed-budget A fits

Best native inspiratory MAE checkpoint and authoritative optimizer/RNG last state are saved each epoch. Each arm runs200 epochs independently. No curve auxiliary loss, combined arm or ensemble-weight tuning is introduced.


In [ ]:
def record_prediction(name,phase,prediction,recipe,partner,extra):
    prediction=np.asarray(prediction,dtype=np.float64);assert prediction.shape==TRUTH.shape and np.isfinite(prediction).all()
    partner_prediction=P_DEEP if partner=='deep' else P_COMPACT;pair=.5*(prediction+partner_prediction);error=np.abs(prediction-TRUTH);pair_error=np.abs(pair-TRUTH)
    prediction_path=OUT/('A_'+name+'_predictions.npy');save_array(prediction_path,prediction)
    counts=MASK.sum(axis=1);sums=np.add.reduceat(pair_error,np.r_[0,np.cumsum(counts)[:-1]])
    r=dict(name=name,phase=phase,recipe=recipe,partner=partner,weights=[.5,.5],seed=SEED,training_breaths=len(GROUPS['A_fit']),validation_breaths=len(GROUPS['A_validation']),validation_mae=float(error.mean()),pair_validation_mae=float(pair_error.mean()),pair_median_absolute_error=float(np.median(pair_error)),pair_median_breath_MAE=float(np.median(sums/counts)),pair_p90_absolute_error=float(np.quantile(pair_error,.9)),prediction_file=str(prediction_path),prediction_sha256=ah(prediction),row_ids_sha256=ah(VAL_IDS),design_sha256=sig(design),**extra)
    write_json(OUT/('record_A_'+name+'.json'),r);RECORDS[name]=r
    rc=np.repeat(RC[GROUPS['A_validation']],80,axis=0)[MASK.ravel()];table=pd.DataFrame(dict(R=rc[:,0],C=rc[:,1],absolute_error=pair_error,signed_error=pair-TRUTH))
    table.groupby(['R','C']).agg(points=('absolute_error','size'),mae=('absolute_error','mean'),total_absolute_error=('absolute_error','sum'),bias=('signed_error','mean')).to_csv(OUT/('A_'+name+'_pair_by_rc.csv'))
    step_grid=np.broadcast_to(np.arange(80),MASK.shape)[MASK]
    inspiration_lengths=MASK.sum(axis=1)
    phase_grid=np.arange(80)[None,:]/np.maximum(inspiration_lengths[:,None]-1,1)
    phase=phase_grid[MASK]
    phase_table=pd.DataFrame(dict(step=step_grid,phase_bin=np.minimum((phase*5).astype(int),4),absolute_error=pair_error,signed_error=pair-TRUTH))
    for key in ['step','phase_bin']:
        phase_table.groupby(key).agg(points=('absolute_error','size'),mae=('absolute_error','mean'),total_absolute_error=('absolute_error','sum'),bias=('signed_error','mean')).to_csv(OUT/('A_'+name+'_pair_by_'+key+'.csv'))
    pd.DataFrame([{k:r.get(k) for k in ['name','phase','validation_mae','pair_validation_mae','pair_median_absolute_error','pair_p90_absolute_error','elapsed_seconds','best_epoch']} for r in RECORDS.values()]).sort_values('pair_validation_mae').to_csv(OUT/'results.csv',index=False)
    event('candidate_complete',name=name,individual_A_MAE=r['validation_mae'],pair_A_MAE=r['pair_validation_mae']);return r

def reusable_record(name,recipe):
    path=OUT/('record_A_'+name+'.json')
    if not path.exists():return None
    r=json.loads(path.read_text());assert RESUME and r['design_sha256']==sig(design) and r['recipe']==recipe
    assert r['row_ids_sha256']==ah(VAL_IDS) and ah(np.load(local(r['prediction_file'])))==r['prediction_sha256']
    if 'checkpoint_path' in r:assert sha(local(r['checkpoint_path']))==r['checkpoint_sha256']
    if 'report_file' in r:assert sha(local(r['report_file']))==r['report_sha256']
    RECORDS[name]=r;return r


In [ ]:
def train_A_candidate_v009(arm):
 cfg=copy.deepcopy(CFG);spec=copy.deepcopy(SPEC);factory=factory_for(arm);loss_kind=LOSS_KINDS[arm]
 recipe=dict(model_spec=spec,train_cfg=cfg,seed=SEED,feature_mode='combined',arm=arm,loss_kind=loss_kind,component_hashes=COMPONENTS,architecture_details=(dict(backend='shifted_linear',channels=96,blocks=6,kernel=3,convolutions_per_block=2,dilations=[1,2,4,8,16,32],receptive_field=253) if arm=='full_context_tcn' else dict(offset_hidden=16,local_hidden=16,centered_over='known_inspiration',extra_parameters=12353)))
 old=reusable_record(arm,recipe)
 if old:return old
 directory=OUT/arm;directory.mkdir(exist_ok=True);checkpoint=directory/'model.pt';fit=GROUPS['A_fit'];val=GROUPS['A_validation'];event('candidate_started',name=arm,recipe=recipe)
 report=fit_validate_v008(X[fit],Y[fit],UOUT[fit],X[val],Y[val],UOUT[val],spec,cfg,SEED,checkpoint,epoch_log_path=directory/'epochs.jsonl',feature_names=NAMES,device='cuda',resume=RESUME and _last_path(checkpoint).exists(),model_factory=factory,loss_kind=loss_kind,arm=arm,source_fingerprints=COMPONENTS)
 assert report['completed'] and report['executed_epochs']==200;write_json(directory/'report.json',report)
 prediction=predict_checkpoint_v008(checkpoint,X[val],device='cuda',batch_size=cfg['prediction_batch_size'],model_factory=factory,expected_arm=arm,source_fingerprints=COMPONENTS)[MASK].astype(np.float64)
 assert abs(float(np.abs(prediction-TRUTH).mean())-report['validation_mae'])<1e-6
 return record_prediction(arm,arm,prediction,recipe,'deep',dict(checkpoint_path=str(checkpoint),checkpoint_sha256=sha(checkpoint),report_file=str(directory/'report.json'),report_sha256=sha(directory/'report.json'),best_epoch=report['best_epoch'],elapsed_seconds=report['elapsed_seconds'],training_eval_mae=report['training_eval_mae'],parameter_count=report['parameter_count'],selected_lr_schedule=report['selected_lr_schedule'],executed_lr_schedule=report['executed_lr_schedule']))
if RUN:
 for arm in ARMS:train_A_candidate_v009(arm)


## Frozen screening decision

Whole-breath bootstrap retains the true point-weighted MAE. Secondary median and tail metrics cannot override the primary rule. A failure does not justify changing thresholds or searching a runner-up on confirmation outcomes. The current best CSV and receipt remain preserved.


In [ ]:
def paired_interval(new_error,old_error,counts):
    sums=np.add.reduceat(new_error-old_error,np.r_[0,np.cumsum(counts)[:-1]]);rng=np.random.default_rng(20261201);draws=[]
    for i in range(0,1000,100):
        take=rng.integers(0,len(counts),size=(100,len(counts)));draws.extend((sums[take].sum(axis=1)/counts[take].sum(axis=1)).tolist())
    lo,hi=np.quantile(draws,[.025,.975]);return dict(difference=float((new_error-old_error).mean()),bootstrap_95_low=float(lo),bootstrap_95_high=float(hi),unit='whole breath',repeats=1000,point_count_weighted=True)
if RUN:
    baseline_error=np.abs(P_REFERENCE-TRUTH);summary=[]
    for name,r in RECORDS.items():
        p=np.load(local(r['prediction_file']));partner=P_DEEP if r['partner']=='deep' else P_COMPACT;pair=.5*(p+partner);error=np.abs(pair-TRUTH);ci=paired_interval(error,baseline_error,MASK.sum(axis=1));gain=float(baseline_error.mean()-error.mean());eligible=bool(gain>=.003 and ci['bootstrap_95_high']<0 )
        summary.append(dict(name=name,individual_A_MAE=r['validation_mae'],fixed_pair_A_MAE=float(error.mean()),gain=gain,interval=ci,eligible_for_fresh_grouped_confirmation=eligible,record_sha256=sha(OUT/('record_A_'+name+'.json'))))
    eligible=[r for r in summary if r['eligible_for_fresh_grouped_confirmation']];selected=min(eligible,key=lambda r:(r['fixed_pair_A_MAE'],r['name'])) if eligible else None
    write_json(OUT/'screen_summary.json',dict(status='complete',reference_A_pair_MAE=float(baseline_error.mean()),results=summary,selected_for_confirmation=selected,minimum_A_gain=.003,confirmation_not_performed=True,confirmation_design='fresh grouped resampling split inside original Afit; independent cold fits, frozen A choice and weights; C/D is confirmation only',official_best_MAE=.1855,new_submission_performed=False,standing_approval=True))
    event('v009_A_screens_complete',selected=selected);fcntl.flock(LOCK,fcntl.LOCK_UN);LOCK.close()
if not RUN and (OUT/'screen_summary.json').exists():print((OUT/'screen_summary.json').read_text())
